# Disciplina 4 — Pesquisa Operacional
## Problema 1: Maximização de Lucro com PuLP

> ⚠️ **Regra inegociável da disciplina**: uso de **Excel Solver é proibido**. Toda otimização aqui é feita com a biblioteca **PuLP** (Programação Linear em Python).

### Cenário (Barbearia Invictus)

A barbearia tem 4 barbeiros disponíveis no sábado, totalizando **40 horas produtivas**. Pode oferecer 3 tipos de pacote:

| Pacote | Duração | Lucro unitário |
|---|---:|---:|
| Corte simples | 30 min | R$ 25 |
| Corte + Barba | 55 min | R$ 45 |
| Pacote Premium (com pigmentação) | 90 min | R$ 80 |

### Restrições do mundo real

- **Tempo total** dos barbeiros: 40h × 60min = **2400 min**
- **Pigmentação**: estoque limitado a 10 atendimentos premium/dia
- **Mix comercial**: pelo menos 30% dos slots devem ser "corte simples" (compromisso de fluxo de novos clientes)

### Pergunta

Quantos atendimentos de cada tipo aceitar para **maximizar o lucro**?

In [1]:
# !pip install pulp --quiet
from pulp import (
    LpInteger, LpMaximize, LpProblem, LpStatus, LpVariable,
    PULP_CBC_CMD, value
)

## 1. Modelagem matemática

**Variáveis de decisão** (todas inteiras, ≥ 0):

- $x_C$ = quantidade de Cortes simples
- $x_B$ = quantidade de Cortes + Barba
- $x_P$ = quantidade de Pacotes Premium

**Função objetivo (maximizar):**

$$Z = 25 x_C + 45 x_B + 80 x_P$$

**Restrições:**

$$30 x_C + 55 x_B + 90 x_P \leq 2400 \quad \text{(tempo total em minutos)}$$

$$x_P \leq 10 \quad \text{(estoque premium)}$$

$$x_C \geq 0{,}3 \cdot (x_C + x_B + x_P) \quad \text{(mix mínimo de corte simples)}$$

In [2]:
def resolver(horas_disponiveis: int = 40, max_premium: int = 10) -> dict:
    minutos_total = horas_disponiveis * 60

    model = LpProblem('Max_Lucro_Sabado', LpMaximize)

    xC = LpVariable('Corte_Simples', lowBound=0, cat=LpInteger)
    xB = LpVariable('Corte_Barba',   lowBound=0, cat=LpInteger)
    xP = LpVariable('Premium',       lowBound=0, cat=LpInteger)

    model += 25 * xC + 45 * xB + 80 * xP, 'Lucro_Total'
    model += 30 * xC + 55 * xB + 90 * xP <= minutos_total, 'Restricao_Tempo'
    model += xP <= max_premium, 'Restricao_Premium'
    model += xC >= 0.3 * (xC + xB + xP), 'Restricao_Mix_Corte'

    model.solve(PULP_CBC_CMD(msg=False))
    return {
        'status': LpStatus[model.status],
        'horas_disponiveis': horas_disponiveis,
        'corte_simples': int(value(xC) or 0),
        'corte_barba':   int(value(xB) or 0),
        'premium':       int(value(xP) or 0),
        'lucro_total':   float(value(model.objective) or 0.0),
    }

## 2. Cenário Base — 4 barbeiros, dia inteiro

In [3]:
import pandas as pd

base = resolver(horas_disponiveis=40)
pd.DataFrame([base]).T.rename(columns={0: 'valor'})

,valor
status,Optimal
horas_disponiveis,40
corte_simples,50
corte_barba,0
premium,10
lucro_total,2050.0


## 3. Análise de Cenário (What-If) — *obrigatório pelo guia*

**Pergunta da gerência:** *"E se um barbeiro adoecer e perdermos 25% das horas?"*

In [4]:
crise = resolver(horas_disponiveis=30)
comparacao = pd.DataFrame([base, crise], index=['Base (40h)', 'Crise (30h)'])
comparacao[['horas_disponiveis', 'corte_simples', 'corte_barba', 'premium', 'lucro_total']]

,horas_disponiveis,corte_simples,corte_barba,premium,lucro_total
Base (40h),40,50,0,10,2050.0
Crise (30h),30,30,0,10,1550.0


In [5]:
perda = base['lucro_total'] - crise['lucro_total']
perda_pct = (perda / base['lucro_total']) * 100
print(f'IMPACTO DO IMPREVISTO: -R$ {perda:,.2f} de lucro ({perda_pct:.1f}% do potencial)')

IMPACTO DO IMPREVISTO: -R$ 500.00 de lucro (24.4% do potencial)


## 4. Análise Crítica — parte 1/2 (Maximização)

> A pergunta obrigatória do guia (item 4.3) é **conjunta** para os dois
> problemas. A resposta consolidada está em
> [`docs/d4_pesquisa_operacional.md`](../docs/d4_pesquisa_operacional.md) § 4.3.

**Os resultados matemáticos fazem sentido para a barbearia?** Sim. O
solver alocou os **10 pacotes premium permitidos** (porque o lucro/minuto
é maior — R$ 0,89/min do premium vs R$ 0,82/min do combo) e completou os
30% mínimos com cortes simples para satisfazer o compromisso comercial
de fluxo de novos clientes. O lucro base de **R$ 2.050,00** representa
o teto teórico em condições normais.

**Como a Análise What-If ajuda a gerência?** Mostrou que a perda de 25%
das horas (1 barbeiro fora) corta o lucro em **24,4%**, evidenciando
que a empresa opera **muito perto da capacidade máxima** sem folga.
Isso conecta diretamente ao **Plano de Riscos do TAP** (Risco R3 — *"Membro
fundamental da equipe adoecer"*) e justifica três mitigações:

1. Manter um **barbeiro reserva** (folga operacional);
2. Permitir **overbooking controlado de 10%** em horários de pico;
3. Priorizar a **compra de produtos para pacotes premium** (já que o solver os escolhe primeiro).